In [ ]:
def main(datasources, start_date, end_date):
    """AI05：成交颗粒度状态模型；固定2019—2022训练并预测传入区间。"""
    import time
    import numpy as np
    import pandas as pd
    import dai
    import xgboost as xgb
    import structlog

    logger = structlog.get_logger()

    IMPLEMENTATION_VERSION = "AI05_TRADE_GRANULARITY_V1_PUBLIC"
    TRAIN_START = "2019-01-01 00:00:00"
    TRAIN_END = "2024-12-31 23:59:59"
    TRAIN_BAR_TABLE = "bigalpha_2026_stock_bar1m"
    PUBLIC_TRAINING_THROUGH_2024 = True

    # 与AI02的盘口/八段价格路径不同，本模型的核心信息来自deal_number：
    # 每笔成交规模、成交笔数在日内的集中与迁移，以及这些量相对自身历史的异常。
    current_feature_columns = [
        "daily_log_value_per_deal",
        "daily_log_share_per_deal",
        "value_size_dispersion",
        "share_size_dispersion",
        "value_size_upper_tail",
        "late_early_value_shift",
        "late_early_share_shift",
        "value_size_time_trend",
        "deal_intensity_time_trend",
        "up_down_value_size_gap",
        "large_trade_signed_pressure",
        "large_trade_amount_share",
        "late_deal_share",
        "late_amount_share",
        "deal_concentration",
        "active_minute_share",
    ]
    historical_feature_columns = [
        "value_size_surprise_20d",
        "share_size_surprise_20d",
        "deal_count_surprise_20d",
        "concentration_surprise_20d",
        "late_shift_surprise_20d",
        "dispersion_surprise_20d",
        "value_size_5d_20d_gap",
        "large_trade_pressure_5d",
    ]
    feature_columns = current_feature_columns + historical_feature_columns
    style_columns = [
        "daily_return",
        "momentum_5",
        "momentum_20",
        "volatility_20",
        "log_total_amount",
        "log_total_deals",
    ]

    def query_daily_trade_granularity(bar_table, sd, ed):
        """在SQL内把分钟成交笔数与成交规模聚合成日频状态。"""
        sql = f"""
        WITH raw_all AS (
            SELECT
                date,
                instrument,
                CAST(date_trunc('day', date) AS DATE) AS trading_day,
                CAST(strftime(date, '%H%M') AS INTEGER) AS hhmm,
                CAST(close AS DOUBLE) AS close,
                CAST(volume AS DOUBLE) AS minute_volume,
                CAST(amount AS DOUBLE) AS minute_amount,
                CAST(deal_number AS DOUBLE) AS minute_deals
            FROM {bar_table}
            WHERE CAST(strftime(date, '%H%M') AS INTEGER) BETWEEN 931 AND 1500
              AND close > 0
        ), price_daily AS (
            SELECT
                trading_day,
                instrument,
                ARG_MIN(close, date) AS first_close,
                ARG_MAX(close, date) AS last_close
            FROM raw_all
            GROUP BY trading_day, instrument
        ), continuous_raw AS (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY trading_day, instrument ORDER BY date
                ) AS minute_index,
                LAG(close) OVER (
                    PARTITION BY trading_day, instrument ORDER BY date
                ) AS previous_close
            FROM raw_all
            WHERE hhmm BETWEEN 931 AND 1457
              AND minute_volume >= 0
              AND minute_amount >= 0
              AND minute_deals >= 0
        ), minute_features AS (
            SELECT
                *,
                CASE WHEN previous_close > 0
                    THEN LN(close / previous_close) END AS minute_return,
                CASE WHEN minute_deals > 0
                    THEN LN(1.0 + minute_volume / minute_deals) END
                    AS log_share_per_deal,
                CASE WHEN minute_deals > 0
                    THEN LN(1.0 + minute_amount / minute_deals) END
                    AS log_value_per_deal,
                LN(1.0 + minute_deals) AS log_deals
            FROM continuous_raw
        ), minute_enriched AS (
            SELECT
                *,
                QUANTILE_CONT(log_value_per_deal, 0.75) OVER (
                    PARTITION BY trading_day, instrument
                ) AS value_size_q75
            FROM minute_features
        ), daily_trade AS (
            SELECT
                trading_day,
                instrument,
                COUNT(*) AS observations,
                COUNT(*) FILTER (WHERE minute_deals > 0) AS active_minutes,
                SUM(minute_volume) AS total_volume,
                SUM(minute_amount) AS total_amount,
                SUM(minute_deals) AS total_deals,
                CASE WHEN SUM(minute_deals) > 0
                    THEN LN(1.0 + SUM(minute_amount) / SUM(minute_deals)) END
                    AS daily_log_value_per_deal,
                CASE WHEN SUM(minute_deals) > 0
                    THEN LN(1.0 + SUM(minute_volume) / SUM(minute_deals)) END
                    AS daily_log_share_per_deal,
                STDDEV_SAMP(log_value_per_deal) AS value_size_dispersion,
                STDDEV_SAMP(log_share_per_deal) AS share_size_dispersion,
                QUANTILE_CONT(log_value_per_deal, 0.90)
                    - QUANTILE_CONT(log_value_per_deal, 0.50)
                    AS value_size_upper_tail,
                AVG(log_value_per_deal) FILTER (WHERE hhmm BETWEEN 1400 AND 1457)
                    - AVG(log_value_per_deal) FILTER (WHERE hhmm BETWEEN 931 AND 1030)
                    AS late_early_value_shift,
                AVG(log_share_per_deal) FILTER (WHERE hhmm BETWEEN 1400 AND 1457)
                    - AVG(log_share_per_deal) FILTER (WHERE hhmm BETWEEN 931 AND 1030)
                    AS late_early_share_shift,
                CORR(CAST(minute_index AS DOUBLE), log_value_per_deal)
                    AS value_size_time_trend,
                CORR(CAST(minute_index AS DOUBLE), log_deals)
                    AS deal_intensity_time_trend,
                AVG(log_value_per_deal) FILTER (WHERE minute_return > 0)
                    - AVG(log_value_per_deal) FILTER (WHERE minute_return < 0)
                    AS up_down_value_size_gap,
                CASE WHEN SUM(minute_amount) FILTER (
                        WHERE log_value_per_deal >= value_size_q75
                    ) > 0
                    THEN SUM(SIGN(minute_return) * minute_amount) FILTER (
                        WHERE log_value_per_deal >= value_size_q75
                    ) / SUM(minute_amount) FILTER (
                        WHERE log_value_per_deal >= value_size_q75
                    ) END AS large_trade_signed_pressure,
                CASE WHEN SUM(minute_amount) > 0
                    THEN SUM(minute_amount) FILTER (
                        WHERE log_value_per_deal >= value_size_q75
                    ) / SUM(minute_amount) END AS large_trade_amount_share,
                CASE WHEN SUM(minute_deals) > 0
                    THEN SUM(minute_deals) FILTER (WHERE hhmm BETWEEN 1400 AND 1457)
                         / SUM(minute_deals) END AS late_deal_share,
                CASE WHEN SUM(minute_amount) > 0
                    THEN SUM(minute_amount) FILTER (WHERE hhmm BETWEEN 1400 AND 1457)
                         / SUM(minute_amount) END AS late_amount_share,
                CASE WHEN SUM(minute_deals) > 0
                    THEN SUM(POWER(minute_deals, 2)) / POWER(SUM(minute_deals), 2)
                    END AS deal_concentration,
                COUNT(*) FILTER (WHERE minute_deals > 0) * 1.0 / COUNT(*)
                    AS active_minute_share
            FROM minute_enriched
            GROUP BY trading_day, instrument
            HAVING COUNT(*) >= 180
        )
        SELECT
            CAST(trade.trading_day AS DATETIME) AS date,
            trade.instrument,
            price.first_close,
            price.last_close,
            trade.observations,
            trade.total_volume,
            trade.total_amount,
            trade.total_deals,
            trade.daily_log_value_per_deal,
            trade.daily_log_share_per_deal,
            trade.value_size_dispersion,
            trade.share_size_dispersion,
            trade.value_size_upper_tail,
            trade.late_early_value_shift,
            trade.late_early_share_shift,
            trade.value_size_time_trend,
            trade.deal_intensity_time_trend,
            trade.up_down_value_size_gap,
            trade.large_trade_signed_pressure,
            trade.large_trade_amount_share,
            trade.late_deal_share,
            trade.late_amount_share,
            trade.deal_concentration,
            trade.active_minute_share
        FROM daily_trade AS trade
        INNER JOIN price_daily AS price
          ON trade.trading_day = price.trading_day
         AND trade.instrument = price.instrument
        -- 联表后两侧都有 instrument，必须显式限定，避免 DuckDB Binder 歧义。
        ORDER BY trade.trading_day, trade.instrument
        """
        frame = dai.query(
            sql,
            filters={"date": [sd, ed]},
            compression=True,
        ).df()
        frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
        frame["instrument"] = frame["instrument"].astype(str)
        numeric_columns = [c for c in frame.columns if c not in {"date", "instrument"}]
        for column in numeric_columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce", downcast="float")
        frame[numeric_columns] = frame[numeric_columns].replace([np.inf, -np.inf], np.nan)
        return frame.sort_values(["instrument", "date"]).reset_index(drop=True)

    def query_chunked(bar_table, sd, ed):
        """按自然年查询，避免四年分钟数据在一次SQL中形成过高内存峰值。"""
        requested_start = pd.to_datetime(sd)
        requested_end = pd.to_datetime(ed)
        parts = []
        chunk_start = requested_start
        while chunk_start <= requested_end:
            year_end = pd.Timestamp(
                year=chunk_start.year,
                month=12,
                day=31,
                hour=23,
                minute=59,
                second=59,
            )
            chunk_end = min(year_end, requested_end)
            logger.info(
                "AI05年度分块查询",
                chunk_start=str(chunk_start),
                chunk_end=str(chunk_end),
            )
            parts.append(query_daily_trade_granularity(bar_table, chunk_start, chunk_end))
            chunk_start = pd.Timestamp(year=chunk_start.year + 1, month=1, day=1)
        if not parts:
            return pd.DataFrame()
        combined = pd.concat(parts, ignore_index=True, copy=False)
        combined = combined.drop_duplicates(["date", "instrument"], keep="last")
        return combined.sort_values(["instrument", "date"]).reset_index(drop=True)

    def rolling_previous(frame, column, window, minimum):
        return frame.groupby("instrument", sort=False)[column].transform(
            lambda values: values.shift(1).rolling(window, min_periods=minimum).mean()
        )

    def rolling_previous_std(frame, column, window, minimum):
        return frame.groupby("instrument", sort=False)[column].transform(
            lambda values: values.shift(1).rolling(window, min_periods=minimum).std()
        )

    def add_history_styles_and_label(frame, include_label):
        """历史异常只引用t-1及以前；未来收益只在训练标签中出现。"""
        enriched = frame.copy()
        grouped_close = enriched.groupby("instrument", sort=False)["last_close"]
        previous_close = grouped_close.shift(1)
        enriched["daily_return"] = enriched["last_close"] / previous_close - 1.0
        enriched["momentum_5"] = enriched["last_close"] / grouped_close.shift(5) - 1.0
        enriched["momentum_20"] = enriched["last_close"] / grouped_close.shift(20) - 1.0
        enriched["volatility_20"] = (
            enriched.groupby("instrument", sort=False)["daily_return"]
            .transform(lambda values: values.rolling(20, min_periods=10).std())
        )
        enriched["log_total_amount"] = np.log1p(enriched["total_amount"].clip(lower=0))
        enriched["log_total_deals"] = np.log1p(enriched["total_deals"].clip(lower=0))

        surprise_map = {
            "value_size_surprise_20d": "daily_log_value_per_deal",
            "share_size_surprise_20d": "daily_log_share_per_deal",
            "deal_count_surprise_20d": "log_total_deals",
            "concentration_surprise_20d": "deal_concentration",
            "late_shift_surprise_20d": "late_early_value_shift",
            "dispersion_surprise_20d": "value_size_dispersion",
        }
        for output_column, source_column in surprise_map.items():
            history_mean = rolling_previous(enriched, source_column, 20, 10)
            history_std = rolling_previous_std(enriched, source_column, 20, 10)
            enriched[output_column] = (
                enriched[source_column] - history_mean
            ) / (history_std.abs() + 1e-6)

        value_mean_5 = rolling_previous(
            enriched, "daily_log_value_per_deal", 5, 3
        )
        value_mean_20 = rolling_previous(
            enriched, "daily_log_value_per_deal", 20, 10
        )
        enriched["value_size_5d_20d_gap"] = value_mean_5 - value_mean_20
        enriched["large_trade_pressure_5d"] = rolling_previous(
            enriched, "large_trade_signed_pressure", 5, 3
        )

        if include_label:
            next_close = grouped_close.shift(-1)
            enriched["raw_label"] = next_close / enriched["last_close"] - 1.0
        return enriched

    def rank_columns(frame):
        ranked = frame.copy()
        rank_sources = list(dict.fromkeys(feature_columns + style_columns))
        for column in rank_sources:
            ranked[column] = pd.to_numeric(ranked[column], errors="coerce")
            ranked[column] = ranked.groupby("date", sort=False)[column].rank(
                method="average", pct=True
            ) - 0.5
        return ranked

    def residualize_training_labels(frame):
        """逐日剔除常见量价风格，训练目标是剩余T+1横截面排序。"""
        target = frame.groupby("date", sort=False)["raw_label"].rank(
            method="average", pct=True
        ) - 0.5
        residual = pd.Series(np.nan, index=frame.index, dtype=float)
        for _, positions in frame.groupby("date", sort=False).groups.items():
            index = np.asarray(list(positions))
            valid = target.loc[index].notna().to_numpy()
            if int(valid.sum()) < 50:
                continue
            styles = frame.loc[index, style_columns].fillna(0.0).to_numpy(dtype=float)
            design = np.column_stack([np.ones(len(index)), styles])
            response = target.loc[index].to_numpy(dtype=float)
            beta, *_ = np.linalg.lstsq(design[valid], response[valid], rcond=None)
            residual.loc[index[valid]] = response[valid] - design[valid] @ beta
        output = frame.copy()
        output["label"] = residual.groupby(output["date"]).rank(
            method="average", pct=True
        ) - 0.5
        return output

    def align_to_pool(frame, sd, ed, how):
        pool = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={"date": [sd, ed]},
        ).df()
        pool["date"] = pd.to_datetime(pool["date"], errors="coerce").dt.normalize()
        pool["instrument"] = pool["instrument"].astype(str)
        pool = pool.drop_duplicates(["date", "instrument"], keep="last")
        bounded = frame[
            (frame["date"] >= pd.to_datetime(sd).normalize())
            & (frame["date"] <= pd.to_datetime(ed).normalize())
        ]
        return pd.merge(pool, bounded, on=["date", "instrument"], how=how)

    def build_dataset(bar_table, sd, ed, include_label, pool_join):
        started = time.time()
        query_start = pd.to_datetime(sd) - pd.Timedelta(days=45)
        raw = query_chunked(bar_table, query_start, ed)
        enriched = add_history_styles_and_label(raw, include_label)
        ranked = rank_columns(enriched)
        aligned = align_to_pool(ranked, sd, ed, pool_join)
        if include_label:
            aligned = residualize_training_labels(aligned)
        logger.info(
            "AI05数据集完成",
            start=str(sd),
            end=str(ed),
            rows=len(aligned),
            elapsed=round(time.time() - started, 2),
        )
        return aligned.sort_values(["date", "instrument"]).reset_index(drop=True)

    logger.info(
        "AI05开始",
        version=IMPLEMENTATION_VERSION,
        train_start=TRAIN_START,
        train_end=TRAIN_END,
        prediction_start=start_date,
        prediction_end=end_date,
        public_training_through_2024=PUBLIC_TRAINING_THROUGH_2024,
    )
    train_data = build_dataset(
        TRAIN_BAR_TABLE,
        TRAIN_START,
        TRAIN_END,
        include_label=True,
        pool_join="inner",
    ).dropna(subset=["label"]).reset_index(drop=True)
    if len(train_data) < 100000:
        raise ValueError(f"AI05训练样本不足: {len(train_data)}")

    train_matrix = train_data[feature_columns].fillna(0.0).to_numpy(dtype="float32")
    train_target = train_data["label"].to_numpy(dtype="float32")
    age_days = (
        pd.to_datetime(TRAIN_END).normalize() - train_data["date"]
    ).dt.days.to_numpy(dtype="float32")
    sample_weight = np.exp(-age_days / 730.0).astype("float32")
    sample_weight = np.maximum(sample_weight, 0.15)

    model = xgb.XGBRegressor(
        n_estimators=180,
        max_depth=4,
        learning_rate=0.035,
        min_child_weight=50,
        subsample=0.75,
        colsample_bytree=0.80,
        reg_alpha=0.25,
        reg_lambda=5.0,
        max_bin=128,
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=-1,
        random_state=20260722,
    )
    fit_started = time.time()
    logger.info(
        "AI05训练模型",
        samples=len(train_data),
        features=len(feature_columns),
    )
    model.fit(train_matrix, train_target, sample_weight=sample_weight)
    logger.info("AI05模型完成", elapsed=round(time.time() - fit_started, 2))
    del train_matrix, train_target, sample_weight, age_days, train_data

    prediction_data = build_dataset(
        datasources["bar1m"],
        start_date,
        end_date,
        include_label=False,
        pool_join="left",
    )
    prediction_matrix = prediction_data[feature_columns].fillna(0.0).to_numpy(
        dtype="float32"
    )
    prediction_data["raw_prediction"] = model.predict(prediction_matrix)
    prediction_data["factor"] = prediction_data.groupby("date", sort=False)[
        "raw_prediction"
    ].rank(method="average", pct=True) - 0.5
    prediction_data["factor"] = pd.to_numeric(
        prediction_data["factor"], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)
    prediction_data["factor"] = prediction_data.groupby("date", sort=False)[
        "factor"
    ].transform(lambda values: values.fillna(values.median())).fillna(0.0)

    result = prediction_data[["date", "instrument", "factor"]].copy()
    result = result.drop_duplicates(["date", "instrument"], keep="last")
    result = result.sort_values(["date", "instrument"]).reset_index(drop=True)
    logger.info("AI05完成", rows=len(result), missing=int(result["factor"].isna().sum()))
    return result[["date", "instrument", "factor"]]
